In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import pickle

In [2]:
# Charger et parser le fichier XML
tree = ET.parse('/home/galadriel/dr_benchmark/raw_data/product6.xml')
root = tree.getroot()

# Initialiser une liste pour stocker les informations
data = []

# Parcourir tous les DrugRegulatoryStatus dans le fichier XML
for drug_status in root.findall('DrugRegulatoryStatusList/DrugRegulatoryStatus'):
    # Extraire l'ATC code (s'il existe)
    # atc_code = drug_status.findtext('ATCCode', default='')
    
    # Extraire les informations de chaque Substance
    for substance_association in drug_status.findall('SubstanceDrugRegulatoryStatusAssociationList/SubstanceDrugRegulatoryStatusAssociation'):
        substance = substance_association.find('Substance')
        
        code = substance.findtext('Code', default='')
        chemical_name = substance.findtext('ChemicalName', default='')
        name = substance.findtext('Name', default='')
        
        # Extraire les OrphaCodes et noms des maladies associés
        for disorder in drug_status.findall('DisorderList/Disorder'):
            orpha_code = disorder.findtext('OrphaCode', default='')
            disorder_name = disorder.findtext('Name', default='')
            
            # Extraire les informations de DrugTradeName (s'il y en a)
            trade_names = []
            for trade_name_association in drug_status.findall('DrugTradeNameDrugRegulatoryStatusAssociationList/DrugTradeNameDrugRegulatoryStatusAssociation'):
                trade_name = trade_name_association.findtext('DrugTradeName', default='')
                if trade_name:
                    trade_names.append(trade_name)

            # Si aucun nom commercial n'est trouvé, mettre une chaîne vide
            trade_names_str = ", ".join(trade_names) if trade_names else ''
            
            # Ajouter les informations dans le tableau
            data.append({
                # 'ATCCode': atc_code,
                'Code': code,
                'ChemicalName': chemical_name,
                'Name': name,
                'DrugTradeName': trade_names_str,
                'OrphaCode': orpha_code,
                'DisorderName': disorder_name
            })

# Convertir les données en DataFrame
df = pd.DataFrame(data)
df['Name'] = df['Name'].str.lower()
df['DisorderName'] = df['DisorderName'].str.lower()

In [3]:
print(df.shape)
df = df.drop_duplicates()
print(df.shape)

(5580, 6)
(4915, 6)


In [4]:
tar_kg = pd.read_csv("/home/galadriel/dr_benchmark/knowledge_graph/TarKG_edges_march_17_2026.csv", sep=",", dtype={"from": str, "rel": str, "to": str})

In [5]:
tar_kg["rel_full"] = tar_kg["node1_type"] + " " + tar_kg["rel"] + " " + tar_kg["node2_type"]
tar_kg

,index,from,node1_type,rel,to,node2_type,rel_full
0,162105,DOID:0001816,Disease,is a,DOID:175,Disease,Disease is a Disease
1,318177,DOID:0080189,Disease,is a,DOID:175,Disease,Disease is a Disease
2,940360,DOID:3316,Disease,is a,DOID:175,Disease,Disease is a Disease
3,1127540,DOID:5547,Disease,is a,DOID:175,Disease,Disease is a Disease
4,1221419,DOID:7388,Disease,is a,DOID:175,Disease,Disease is a Disease
...,...,...,...,...,...,...,...
32806462,691478,DOID:14227,Symptom,associated with,CTD:Congenital bilateral aplasia of vas deferens,Disease,Symptom associated with Disease
32806463,691493,DOID:14227,Symptom,associated with,UMLS:C4014454,Disease,Symptom associated with Disease
32806464,691487,DOID:14227,Symptom,associated with,UMLS:C1970187,Disease,Symptom associated with Disease
32806465,691492,DOID:14227,Symptom,associated with,UMLS:C4014449,Disease,Symptom associated with Disease


In [6]:
tar_kg_nodes = pd.read_csv("/home/galadriel/dr_benchmark/knowledge_graph/TarKG_nodes_mapping_march_17_2026.csv")
tar_kg_nodes

,index,unify_id,kind,kgid,dbid,db_source,name,source,kg_index,kg
0,1,"CTD:Acrocallosal syndrome, Schinzel type",Disease,NaN,C2931760,UMLS,"Acrocallosal syndrome, Schinzel type",CTD,tcmKG-82821,tcmKG
1,2,"CTD:Adrenal hyperplasia, congenital, type 5",Disease,NaN,C0268285,UMLS,"Adrenal hyperplasia, congenital, type 5",CTD,tcmKG-78063,tcmKG
2,3,CTD:Alcoholism,Disease,C0001973,C0001973,UMLS,Alcoholism,CTD,MSI-3616,MSI
3,3,CTD:Alcoholism,Disease,C0001973,C0001973,UMLS,Alcoholism,CTD,BioKG-5845,BioKG
4,3,CTD:Alcoholism,Disease,NaN,C0001973,UMLS,Alcoholism,CTD,tcmKG-76666,tcmKG
...,...,...,...,...,...,...,...,...,...,...
1731092,1143309,TCM_Syndrome95,TCM_Syndrome,NaN,TCM_Syndrome95,NaN,accumulation of heat,NaN,tcmKG-11984,tcmKG
1731093,1143310,TCM_Syndrome96,TCM_Syndrome,NaN,TCM_Syndrome96,NaN,febrile symptoms,NaN,tcmKG-11985,tcmKG
1731094,1143311,TCM_Syndrome97,TCM_Syndrome,NaN,TCM_Syndrome97,NaN,cough caused by dryness,NaN,tcmKG-11986,tcmKG
1731095,1143312,TCM_Syndrome98,TCM_Syndrome,NaN,TCM_Syndrome98,NaN,adverse rising of phlegm,NaN,tcmKG-11987,tcmKG


In [7]:
# Construire les dicts de mapping nom -> id pour drug et disease
drug_node_data    = tar_kg_nodes[tar_kg_nodes["kind"] == "Compound"].drop_duplicates(subset="unify_id")
disease_node_data = tar_kg_nodes[tar_kg_nodes["kind"] == "Disease"].drop_duplicates(subset="unify_id")

In [8]:
drug_name_to_id    = dict(zip(drug_node_data["name"].str.lower(),    drug_node_data["unify_id"]))
disease_name_to_id = dict(zip(disease_node_data["name"].str.lower(), disease_node_data["unify_id"]))

In [9]:
# Mapper les entités Orphanet vers les IDs du KG
df["drug_kg_id"]    = df["Name"].str.lower().map(drug_name_to_id)
df["disease_kg_id"] = df["DisorderName"].str.lower().map(disease_name_to_id)

In [10]:
filtered_df = df.dropna(subset=["drug_kg_id", "disease_kg_id"])
orpha_pairs = set(zip(filtered_df["drug_kg_id"], filtered_df["disease_kg_id"]))
print(f"Paires Orphanet matchées : {len(orpha_pairs)}")


Paires Orphanet matchées : 907


In [11]:
# Séparer déjà dans le KG vs manquantes
# On travaille uniquement sur les arêtes "Compound indication Disease"
indication_edges = tar_kg[tar_kg["rel_full"] == "Compound indication Disease"]
kg_indication_pairs = set(zip(indication_edges["from"], indication_edges["to"]))

already_in = orpha_pairs & kg_indication_pairs
missing    = orpha_pairs - kg_indication_pairs
print(f"Déjà dans le KG : {len(already_in)}")
print(f"Manquantes (arête absente, nœuds présents) : {len(missing)}")

Déjà dans le KG : 264
Manquantes (arête absente, nœuds présents) : 643


In [12]:
# 5. Construire le KG final
# Supprimer les arêtes indication qui sont dans orpha_pairs
tar_kg_filtered = tar_kg[
    ~((tar_kg["rel_full"] == "Compound indication Disease") &
      (tar_kg.apply(lambda r: (r["from"], r["to"]) in orpha_pairs, axis=1)))
]


In [13]:
# Construire les nouvelles arêtes orpha_indication (already_in + missing)
new_rows = []
for drug_id, disease_id in orpha_pairs:
    new_rows.append({
        "from":       drug_id,
        "node1_type": "Compound",
        "rel":        "orpha_indication",
        "rel_full":   "Compound orpha_indication Disease",
        "to":         disease_id,
        "node2_type": "Disease"
    })

new_edges = pd.DataFrame(new_rows)
new_edges

,from,node1_type,rel,rel_full,to,node2_type
0,TC319559,Compound,orpha_indication,Compound orpha_indication Disease,DOID:11339,Disease
1,TC421012,Compound,orpha_indication,Compound orpha_indication Disease,DOID:0060655,Disease
2,TC574545,Compound,orpha_indication,Compound orpha_indication Disease,DOID:9119,Disease
3,TC409458,Compound,orpha_indication,Compound orpha_indication Disease,DOID:3347,Disease
4,TC38394,Compound,orpha_indication,Compound orpha_indication Disease,DOID:0081076,Disease
...,...,...,...,...,...,...
902,TC631548,Compound,orpha_indication,Compound orpha_indication Disease,MONDO:0015796,Disease
903,TC359019,Compound,orpha_indication,Compound orpha_indication Disease,DOID:10772,Disease
904,TC120217,Compound,orpha_indication,Compound orpha_indication Disease,DOID:0060768,Disease
905,TC25399,Compound,orpha_indication,Compound orpha_indication Disease,DOID:0070328,Disease


In [14]:
# KG final
tar_kg_final = pd.concat([tar_kg_filtered, new_edges], axis=0, ignore_index=True)

tar_kg_final = tar_kg_final.rename(columns={"rel": "rel_init", "rel_full": "rel"})

# Export
tar_kg_final.to_csv("/home/galadriel/dr_benchmark/knowledge_graph/TarKG_with_orpha_indication.tsv", index=False, sep="\t")

tar_kg_no_orpha = tar_kg_final[tar_kg_final["rel_init"] != "orpha_indication"]
tar_kg_no_orpha.to_csv("/home/galadriel/dr_benchmark/knowledge_graph/TarKG_without_orpha_indication.tsv", index=False, sep="\t")

In [15]:
tar_kg_orpha = tar_kg_final[tar_kg_final["rel_init"] == "orpha_indication"]
tar_kg_orpha.to_csv("/home/galadriel/dr_benchmark/knowledge_graph/TarKG_orpha_indication_only.tsv", index=False, sep="\t")